# Heat diffusion on a plate

*thermodynamics · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/thermodynamics/heat_diffusion)

Fourier's law turns a temperature gradient into a heat flux, and energy conservation turns the divergence of that flux into a change of temperature. On a grid both derivatives are tiny convolution kernels, so one time step is just two conv2d calls over the whole plate.

**Vector form**

$$\mathbf q = -k\,\nabla T,\qquad \rho c\,\partial_t T = -\nabla\cdot\mathbf q = k\,\nabla^2 T$$

**Index (tensor) form**

$$q_i = -k\,\partial_i T,\qquad \partial_t T = -\tfrac{1}{\rho c}\,\partial_i q_i = \alpha\,\partial_i\partial_i T$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $T$ | `T` | (1, 1, H, W) | temperature of every grid cell (batch and channel dims for conv2d) |
| $\partial_i$ | `gradient_kernel(dx)` | (2, 1, 3, 3) | forward-difference stencils, one output channel per direction $i \in \{x, y\}$ |
| $q_i$ | `-k * conv2d(T_padded, grad)` | (1, 2, H+2, W+2) | Fourier's law: heat flux, channel $i$ is the component |
| $\partial_i q_i$ | `conv2d(q, div)` | (1, 1, H, W) | repeated $i$ ⇒ summed: conv2d adds up its 2 input channels — the divergence |
| $\alpha$ | `k / rho_c` | scalar | thermal diffusivity (m²/s) |
| $\partial_t T$ | `dT_dt(T, ...)` | (1, 1, H, W) | right-hand side; forward Euler: $T \leftarrow T + dt\,\partial_t T$ |

Forward-then-backward differences combine into the familiar 5-point Laplacian, so $\partial_i\partial_i T$ is never coded separately: it falls out of composing the gradient and divergence kernels. The explicit scheme is only stable for $\alpha\,dt/dx^2 \le 1/4$ (the "stability number" below). With Dirichlet boundaries the edges are a 20 °C bath and heat drains away; with Neumann (insulated) edges no flux crosses the boundary and the total heat is conserved while the temperature evens out. A sine mode $\sin(\pi x/L)\sin(\pi y/L)$ decays as $\exp(-2\pi^2\alpha t/L^2)$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/thermodynamics/heat_diffusion/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    alpha = params["alpha"] * 1e-6  # mm²/s → m²/s
    X, Y, dx = tensors.grid(params["n_grid"], dev)
    T0 = tensors.hot_spots(X, Y, params["n_spots"])  # (1, 1, n, n)
    rho_c = tensors.heat_capacity(alpha)
    boundary = params["boundary"]

    times, frames = solve.solve(
        T0, tensors.K, rho_c, dx, params["t_max"], params["r"], boundary, tensors.T_EDGE
    )  # (F,), (F, n, n)
    flux = solve.flux_snapshots(frames, tensors.K, dx, boundary, tensors.T_EDGE)  # (F, 2, n, n)

    heat = solve.heat_content(frames, rho_c, dx, tensors.T_EDGE)  # (F,)
    print(f"alpha = {alpha:.2e} m^2/s, dx = {1000 * dx:.2f} mm, stability number alpha dt/dx^2 = {params['r']}")
    print(f"max T: {frames[0].max():.1f} °C -> {frames[-1].max():.1f} °C")
    print(f"heat content changed by {100 * (heat[-1] / heat[0] - 1):+.2f} % ({boundary} edges)")
    t_range = (tensors.T_EDGE, tensors.T_HOT)
    return plot.animate(times, frames, flux, heat, X, Y, t_range, out_dir / "heat_diffusion.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/thermodynamics/heat_diffusion/equation.py
r"""Physics → tensor translation.

Fourier's law says heat flows down the temperature gradient, and energy conservation says
the temperature of a cell changes by the net heat flowing into it:

    \mathbf q = -k \nabla T,        \rho c \, \partial_t T = -\nabla \cdot \mathbf q

In index notation, with i ∈ {x, y} the spatial direction:

    q_i = -k \, \partial_i T,        \partial_t T = -\frac{1}{\rho c} \partial_i q_i = \alpha \, \partial_i \partial_i T

The repeated index i in \partial_i q_i is summed: \partial_x q_x + \partial_y q_y, the divergence.
Putting the two together gives \partial_i \partial_i T = \nabla^2 T, the Laplacian, with α = k / (ρc).

On a grid the derivatives become finite-difference stencils, i.e. small convolution kernels:

    gradient    ∂_i T  →  conv2d(T, GRAD)   1 input channel  → 2 output channels (one per i)
    divergence  ∂_i q_i →  conv2d(q, DIV)    2 input channels → 1 output channel

conv2d sums over its input channels, so the second convolution performs the sum over i for us.
"""

import torch
import torch.nn.functional as F


def gradient_kernel(dx: float) -> torch.Tensor:
    """Forward differences  ∂_x T ≈ (T[y, x+1] − T[y, x]) / dx  (same for y).  Shape (2, 1, 3, 3)."""
    k = torch.zeros(2, 1, 3, 3)
    k[0, 0, 1, 1], k[0, 0, 1, 2] = -1.0, 1.0  # i = x: neighbour to the right
    k[1, 0, 1, 1], k[1, 0, 2, 1] = -1.0, 1.0  # i = y: neighbour below (next row)
    return k / dx


def divergence_kernel(dx: float) -> torch.Tensor:
    """Backward differences  ∂_x q_x ≈ (q_x[y, x] − q_x[y, x−1]) / dx.  Shape (1, 2, 3, 3)."""
    k = torch.zeros(1, 2, 3, 3)
    k[0, 0, 1, 0], k[0, 0, 1, 1] = -1.0, 1.0  # i = x: neighbour to the left
    k[0, 1, 0, 1], k[0, 1, 1, 1] = -1.0, 1.0  # i = y: neighbour above
    return k / dx


def pad_boundary(T: torch.Tensor, boundary: str, T_edge: float) -> torch.Tensor:
    """Add two layers of ghost cells around T (1, 1, H, W) → (1, 1, H+4, W+4).

    dirichlet: ghost cells held at T_edge (the edges are a cold bath).
    neumann:   ghost cells copy the edge cells, so ∂_i T = 0 there and no heat leaks out.
    """
    if boundary == "dirichlet":
        return F.pad(T, (2, 2, 2, 2), mode="constant", value=T_edge)
    return F.pad(T, (2, 2, 2, 2), mode="replicate")


def heat_flux(T_padded: torch.Tensor, k: float, grad: torch.Tensor) -> torch.Tensor:
    r"""q_i = -k ∂_i T.   (1, 1, H+4, W+4) → (1, 2, H+2, W+2): channel i is the flux component."""
    return -k * F.conv2d(T_padded, grad)


def divergence(q: torch.Tensor, div: torch.Tensor) -> torch.Tensor:
    r"""∂_i q_i, summed over i by conv2d.   (1, 2, H+2, W+2) → (1, 1, H, W)."""
    return F.conv2d(q, div)


def dT_dt(T: torch.Tensor, k: float, rho_c: float, grad, div, boundary: str, T_edge: float) -> torch.Tensor:
    r"""∂_t T = -∂_i q_i / (ρc)   (for constant k this is α ∂_i ∂_i T, the 5-point Laplacian)."""
    q = heat_flux(pad_boundary(T, boundary, T_edge), k, grad)
    return -divergence(q, div) / rho_c


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/thermodynamics/heat_diffusion/tensors.py
"""Build the concrete tensors: the grid, the material constants and the initial hot spots."""

import torch

PLATE = 0.1  # side of the square plate (m)
K = 400.0  # thermal conductivity k (W / (m K)), roughly copper
T_EDGE = 20.0  # temperature of the cold edges (°C)
T_HOT = 100.0  # peak temperature of a hot spot (°C)


def grid(n: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor, float]:
    """Cell centres x_j = (j + 1) dx, j = 0..n−1, so the ghost cells sit exactly on the plate edges.

    Returns X, Y of shape (n, n) (row index = y, column index = x) and the spacing dx.
    """
    dx = PLATE / (n + 1)
    x = (torch.arange(n, device=device) + 1) * dx  # (n,)
    Y, X = torch.meshgrid(x, x, indexing="ij")  # (n, n) each
    return X, Y, dx


def heat_capacity(alpha: float) -> float:
    """ρc = k / α  (J / (m³ K)), chosen so the diffusivity matches the requested α."""
    return K / alpha


def hot_spots(X: torch.Tensor, Y: torch.Tensor, n_spots: int) -> torch.Tensor:
    """Initial temperature T(y, x): cold plate plus n_spots Gaussian hot spots on a ring.  Shape (1, 1, n, n).

    conv2d wants (batch, channel, height, width), so we keep two leading singleton dims.
    """
    angle = 2 * torch.pi * torch.arange(n_spots) / n_spots + 0.3  # (S,)
    radius = 0.0 if n_spots == 1 else 0.25 * PLATE
    cx = 0.5 * PLATE + radius * torch.cos(angle)  # (S,) spot centres
    cy = 0.5 * PLATE + radius * torch.sin(angle)
    width = 0.1 * PLATE

    r2 = (X[None] - cx[:, None, None].to(X)) ** 2 + (Y[None] - cy[:, None, None].to(X)) ** 2  # (S, n, n)
    bumps = torch.exp(-r2 / (2 * width**2)).amax(dim=0)  # (n, n) highest bump at each cell, ≤ 1
    T = T_EDGE + (T_HOT - T_EDGE) * bumps
    return T[None, None]


def sine_mode(X: torch.Tensor, Y: torch.Tensor) -> torch.Tensor:
    """T = sin(πx/L) sin(πy/L): the slowest-decaying mode with T = 0 on the edges.  Shape (1, 1, n, n)."""
    return (torch.sin(torch.pi * X / PLATE) * torch.sin(torch.pi * Y / PLATE))[None, None]


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/thermodynamics/heat_diffusion/solve.py
"""Explicit (forward Euler) time stepping of ∂_t T = -∂_i q_i / (ρc)."""

import torch

from .equation import divergence_kernel, dT_dt, gradient_kernel, heat_flux, pad_boundary


def stable_dt(alpha: float, dx: float, r: float) -> float:
    """Forward Euler on the 2D Laplacian is stable only if  r = α dt / dx² ≤ 1/4."""
    if r > 0.25:
        raise ValueError(f"unstable: α dt / dx² = {r} > 1/4")
    return r * dx**2 / alpha


def solve(
    T0: torch.Tensor,
    k: float,
    rho_c: float,
    dx: float,
    t_max: float,
    r: float = 0.2,
    boundary: str = "dirichlet",
    T_edge: float = 0.0,
    n_snapshots: int = 121,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (F,), temperature snapshots (F, H, W)).

    Each step is  T ← T + dt ∂_t T : two convolutions over the whole grid at once.
    """
    dt = stable_dt(k / rho_c, dx, r)
    n_steps = max(1, int(round(t_max / dt)))
    dt = t_max / n_steps  # land exactly on t_max (only makes r slightly smaller)
    grad = gradient_kernel(dx).to(T0)  # (2, 1, 3, 3)
    div = divergence_kernel(dx).to(T0)  # (1, 2, 3, 3)
    keep = set(torch.linspace(0, n_steps, n_snapshots).round().int().tolist())  # steps to record

    T, times, frames = T0, [0.0], [T0[0, 0]]
    for step in range(1, n_steps + 1):
        T = T + dt * dT_dt(T, k, rho_c, grad, div, boundary, T_edge)
        if step in keep:
            times.append(step * dt)
            frames.append(T[0, 0])
    return torch.tensor(times), torch.stack(frames)


def flux_snapshots(frames: torch.Tensor, k: float, dx: float, boundary: str, T_edge: float) -> torch.Tensor:
    """Heat flux q_i for every snapshot, for the arrows in the plot.  (F, H, W) → (F, 2, H, W)."""
    T_padded = pad_boundary(frames[:, None], boundary, T_edge)  # (F, 1, H+4, W+4)  F acts as the batch
    q = heat_flux(T_padded, k, gradient_kernel(dx).to(frames))  # (F, 2, H+2, W+2)
    return q[:, :, 1:-1, 1:-1]  # drop the ghost faces


def heat_content(frames: torch.Tensor, rho_c: float, dx: float, T_edge: float) -> torch.Tensor:
    """Excess thermal energy per unit plate thickness  ∫ ρc (T − T_edge) dA  (J/m).  (F, H, W) → (F,)."""
    return rho_c * (frames - T_edge).sum(dim=(1, 2)) * dx**2


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/thermodynamics/heat_diffusion/plot.py
"""Animate the temperature field with heat-flux arrows, next to the plate's total heat content."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(
    times: torch.Tensor,
    frames: torch.Tensor,
    flux: torch.Tensor,
    heat: torch.Tensor,
    X: torch.Tensor,
    Y: torch.Tensor,
    t_range: tuple[float, float],
    path: Path,
) -> Path:
    T = frames.cpu().numpy()  # (F, H, W)
    q = flux.cpu().numpy()  # (F, 2, H, W)
    t = times.cpu().numpy()
    heat = (heat / heat[0]).cpu().numpy()  # (F,) relative to the start
    x_cm, y_cm = 100 * X.cpu().numpy(), 100 * Y.cpu().numpy()
    side = 100 * (X[0, -1] + X[0, 0]).item()  # plate side in cm
    s = max(1, T.shape[-1] // 8)  # draw one arrow every s cells
    arrow_cm = 0.8 * s * (x_cm[0, 1] - x_cm[0, 0])  # length of the longest arrow

    fig, (ax, ax_q) = plt.subplots(1, 2, figsize=(7, 4), gridspec_kw={"width_ratios": [1.25, 1]})
    image = ax.imshow(T[0], origin="lower", extent=(0, side, 0, side), cmap="inferno", vmin=t_range[0], vmax=t_range[1])
    fig.colorbar(image, ax=ax, label="T (°C)", fraction=0.046, pad=0.03)
    arrows = ax.quiver(
        x_cm[::s, ::s], y_cm[::s, ::s], 0 * x_cm[::s, ::s], 0 * y_cm[::s, ::s],
        color="#93c5fd", angles="xy", scale_units="xy", scale=1 / arrow_cm,
    )  # fmt: skip
    ax.set_xlabel("x (cm)")
    ax.set_ylabel("y (cm)")
    ax.grid(False)
    title = ax.set_title("")

    ax_q.set_xlim(0, t[-1])
    ax_q.set_ylim(0, 1.1)
    ax_q.set_xlabel("t (s)")
    ax_q.set_ylabel("Q(t) / Q(0)")
    ax_q.set_title("heat content ∫ρc(T − T_edge) dA")
    (curve,) = ax_q.plot([], [], lw=2)

    def update(step: int):
        image.set_data(T[step])
        q_max = max(1e-12, float((q[step, 0] ** 2 + q[step, 1] ** 2).max() ** 0.5))
        arrows.set_UVC(q[step, 0, ::s, ::s] / q_max, q[step, 1, ::s, ::s] / q_max)  # rescaled each frame
        curve.set_data(t[: step + 1], heat[: step + 1])
        title.set_text(f"t = {t[step]:.1f} s,  max T = {T[step].max():.0f} °C")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t)), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'alpha': 20.0,
    't_max': 20.0,
    'n_spots': 3,
    'n_grid': 64,
    'r': 0.2,
    'boundary': 'dirichlet'
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.thermodynamics.heat_diffusion")]:
    del sys.modules[name]
main = importlib.import_module("problems.thermodynamics.heat_diffusion.main")
problem_dir = Path("problems/thermodynamics/heat_diffusion")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
